# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rsf-rawnak/FlyRankAI-ML-Internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

**Lane 2 — Refresh / Content Opportunity Scoring** (locked Week 1). Week 4 built a transparent rule baseline (Precision@50 = 0.360 vs. base rate 0.542 — the rule did **not** beat random). This week: a real model, same data, same split, same metric, to see if it can.

## 1. Method choice and why

**Question shape:** "which pages first?" — a ranking problem over a yes/no observed label (`is_declining_label`). Per the method table: *yes/no with an observed label → Logistic Regression, then Random Forest*, evaluated at precision@K because ranking needs scores, not hard labels.

**Chosen methods, in order:**
1. **Logistic Regression** — readable first. Coefficients say plainly which direction each signal pushes risk, and it's the natural upgrade from a hand-written rule: same idea (weighted signals → a score), except the weights are fit instead of guessed.
2. **Random Forest** — only added because the comparison has to earn it. If it doesn't clearly beat Logistic Regression at the same metric, the extra complexity isn't justified (per "does not reward complexity alone").

**Target:** `is_declining_label` (`trend_direction == 'down'`) — observed, not something I defined by a rule.

**Features:** trailing, already-known-today signals only. Excluded on purpose: `trend_direction`, `trend_pct` (the label's own source columns), and `impressions_last_30d` / `clicks_last_30d` / `sessions_last_30d` / `*_prev_30d` — the data dictionary shows `trend_pct` is literally computed from `(impressions_last_30d - impressions_prev_30d) / impressions_prev_30d`, so those columns would hand the model the label in disguise.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

local_path = Path("../../data/raw/content_refresh_anonymized.csv")
raw_url = "https://raw.githubusercontent.com/rsf-rawnak/FlyRankAI-ML-Internship/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(local_path) if local_path.exists() else pd.read_csv(raw_url)

df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
base_rate = df["is_declining_label"].mean()

# Missingness follows content_type (per data dictionary) -- flag, don't blind-fillna
df["has_search_volume"] = df["search_volume"].notna().astype(int)
df["has_word_count"] = df["word_count"].notna().astype(int)
df["has_avg_position"] = (df["avg_position"] > 0).astype(int)  # avg_position==0 means "no data", not rank zero

num_features = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "content_age_days", "days_since_last_update", "ctr", "avg_position",
    "engagement_rate", "scroll_rate", "ai_traffic_pct",
    "has_search_volume", "has_word_count", "has_avg_position",
]
cat_features = ["content_type", "main_intent", "competition_level", "age_tier", "freshness_tier"]

leaked_cols = {"trend_direction", "trend_pct", "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
               "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d", "is_declining_label"}
print("feature/label leak check (should be empty):", set(num_features + cat_features) & leaked_cols)

df["log_impressions_90d"] = np.log1p(df["impressions_90d"])
df["log_clicks_90d"] = np.log1p(df["clicks_90d"])
df["log_sessions_90d"] = np.log1p(df["sessions_90d"])
num_features += ["log_impressions_90d", "log_clicks_90d", "log_sessions_90d"]

# Fill: numeric medians for genuinely missing values (flags above preserve the "why"), 0 is never silently used
for c in ["search_volume", "competition", "cpc", "word_count", "char_count", "scroll_rate"]:
    df[c] = df[c].fillna(df[c].median())
df["avg_position_filled"] = df["avg_position"].replace(0, 100)  # 0 = "not ranked" -> treat as worse than any real rank
num_features = [c if c != "avg_position" else "avg_position_filled" for c in num_features]

print(f"rows: {len(df):,} | base rate: {base_rate:.3f} | numeric features: {len(num_features)} | categorical: {len(cat_features)}")

feature/label leak check (should be empty): set()
rows: 30,000 | base rate: 0.542 | numeric features: 18 | categorical: 5


## 2. Split design

**Grouped by `client_id`.** The data dictionary is explicit that `client_id` is a pseudonym for grouping/splitting only, never a feature — and a random row-level split would let the same client's pages appear in both train and test, so the model could partly memorize client-level quirks instead of learning generalizable signal. An 80/20 **GroupShuffleSplit** on `client_id`, fixed seed, keeps every client entirely on one side. This is the same honest standard the reference pipeline uses (`03_train_model.py`'s "client-holdout split"), and it's the split the Week-4 baseline gets re-scored on below, so the comparison is apples-to-apples.

In [2]:
from sklearn.model_selection import GroupShuffleSplit

X = df[num_features + cat_features].copy()
y = df["is_declining_label"].copy()
groups = df["client_id"].copy()

gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
df_test = df.iloc[test_idx].copy()

overlap = set(groups.iloc[train_idx]) & set(groups.iloc[test_idx])
print(f"train rows: {len(X_train):,} ({groups.iloc[train_idx].nunique()} clients) | "
      f"test rows: {len(X_test):,} ({groups.iloc[test_idx].nunique()} clients)")
print(f"client overlap between train/test (must be empty): {overlap}")
print(f"train base rate: {y_train.mean():.3f} | test base rate: {y_test.mean():.3f}")

train rows: 23,837 (25 clients) | test rows: 6,163 (7 clients)
client overlap between train/test (must be empty): set()
train base rate: 0.550 | test base rate: 0.511


## 3. Train + compare vs my baseline

Same test split, same metric (**precision@50**) as Week 4. The baseline rule (visibility × staleness/CTR-gap flags) is **re-scored here on the held-out test rows only** — Week 4's 0.360 was computed on the full dataset, so re-scoring it on this exact split keeps the comparison honest, not just re-quoting last week's number.

In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

preprocess = ColumnTransformer([
    ("num", StandardScaler(), num_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features),
])

log_reg = Pipeline([("prep", preprocess), ("clf", LogisticRegression(max_iter=1000, random_state=42))])
rand_forest = Pipeline([("prep", preprocess), ("clf", RandomForestClassifier(
    n_estimators=300, max_depth=8, min_samples_leaf=20, random_state=42, n_jobs=-1))])

log_reg.fit(X_train, y_train)
rand_forest.fit(X_train, y_train)

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

K = 50

# --- Week-4 baseline rule, re-scored on this exact test split ---
stale_flag = (df_test["days_since_last_update"] >= 90).astype(int)
ctr_weak_flag = ((df_test["avg_position"] > 0) & (df_test["avg_position"] <= 20) & (df_test["ctr"] < 0.30)).astype(int)
visible_flag = (df_test["impressions_90d"] >= 250).astype(int)
baseline_scores = visible_flag * (1 + stale_flag + ctr_weak_flag) * np.log1p(df_test["impressions_90d"])

logreg_scores = log_reg.predict_proba(X_test)[:, 1]
rf_scores = rand_forest.predict_proba(X_test)[:, 1]

results = pd.DataFrame({
    "method": ["Week-4 rule baseline", "Logistic Regression", "Random Forest"],
    f"precision@{K}": [
        precision_at_k(baseline_scores, y_test, K),
        precision_at_k(logreg_scores, y_test, K),
        precision_at_k(rf_scores, y_test, K),
    ],
    "test_base_rate": [y_test.mean()] * 3,
})
results[f"precision@{K}"] = results[f"precision@{K}"].round(3)
results["test_base_rate"] = results["test_base_rate"].round(3)
print(results.to_string(index=False))

              method  precision@50  test_base_rate
Week-4 rule baseline          0.34           0.511
 Logistic Regression          0.86           0.511
       Random Forest          0.52           0.511


**Reading the table:** both models clear the baseline, and clearly — Logistic Regression: **0.86**, Random Forest: **0.52**, Week-4 rule: **0.34** (test-split base rate 0.511). The simpler model wins outright here, which is the "simplicity is a feature" case the skill points at directly: adding Random Forest's extra complexity did **not** earn its keep on this split — it beat the baseline but lost to plain Logistic Regression by a wide margin. On `n=50`, a 6-point rule-baseline gap (0.34 → the base rate-adjacent RF number) would be shaky, but a 0.34 → 0.86 jump for Logistic Regression is a real, decision-changing gap, not noise.

## 4. Errors and interpretation

What the stronger model leans on (coefficients or feature importance), and three concrete wrong calls read by hand — a metric alone is decoration without this.

In [4]:
# Pick whichever model wins the table above for the deep-dive (adjust if needed after running Section 3)
best_name = results.loc[results[f"precision@{K}"].idxmax(), "method"]
print("best by precision@50:", best_name)

# Logistic Regression coefficients -- direction and rough magnitude
feature_names = (num_features
                  + list(log_reg.named_steps["prep"].named_transformers_["cat"].get_feature_names_out(cat_features)))
coefs = log_reg.named_steps["clf"].coef_[0]
coef_table = pd.DataFrame({"feature": feature_names, "coef": coefs}).sort_values("coef", key=abs, ascending=False).head(10)
print("\nTop 10 Logistic Regression coefficients by |weight|:")
print(coef_table.to_string(index=False))

# Random Forest feature importance
rf_importances = rand_forest.named_steps["clf"].feature_importances_
rf_table = pd.DataFrame({"feature": feature_names, "importance": rf_importances}).sort_values("importance", ascending=False).head(10)
print("\nTop 10 Random Forest feature importances:")
print(rf_table.to_string(index=False))

best by precision@50: Logistic Regression

Top 10 Logistic Regression coefficients by |weight|:
                        feature      coef
            log_impressions_90d  1.017749
                 log_clicks_90d -0.688728
               has_avg_position  0.486767
content_type_comparison article -0.481888
            avg_position_filled -0.471338
            freshness_tier_181+ -0.428547
       main_intent_navigational -0.342058
                 has_word_count  0.326522
               log_sessions_90d -0.280228
                main_intent_nan  0.255186

Top 10 Random Forest feature importances:
            feature  importance
log_impressions_90d    0.189143
avg_position_filled    0.149535
   content_age_days    0.117653
   has_avg_position    0.077560
         char_count    0.045621
         word_count    0.044826
      age_tier_365+    0.037381
                ctr    0.034494
     log_clicks_90d    0.030262
        scroll_rate    0.029679


**Sanity check on the top features:** `log_impressions_90d` leads both models — plausible, since visibility (impressions) is the volume base that makes any subsequent decline observable at all. Logistic Regression's next drivers are `log_clicks_90d` (negative — more clicks, less risk) and `has_avg_position`/`avg_position_filled` (not being ranked, or ranking worse, raises risk). Random Forest leans on the same `avg_position` signal plus `content_age_days` — older content skewing riskier is a sensible, non-suspicious pattern. Nothing here is "suspiciously perfect" (no single feature near-fully explains the label), which is the leakage smell to watch for — it doesn't show up.

In [5]:
# Three concrete wrong cases from the stronger model
rf_probs = rand_forest.predict_proba(X_test)[:, 1]
df_test_scored = df_test.copy()
df_test_scored["rf_score"] = rf_probs
df_test_scored["rf_pred"] = (rf_probs >= 0.5).astype(int)
wrong = df_test_scored[df_test_scored["rf_pred"] != df_test_scored["is_declining_label"]]

false_positives = wrong[wrong["rf_pred"] == 1].sort_values("rf_score", ascending=False).head(2)
false_negatives = wrong[wrong["rf_pred"] == 0].sort_values("rf_score").head(1)
review_cols = ["content_id", "rf_score", "is_declining_label", "days_since_last_update", "avg_position", "ctr", "engagement_rate"]

print("False positives (model said declining, wasn't) -- confident but wrong, worth reading:")
print(false_positives[review_cols].to_string(index=False))
print("\nFalse negative (model said stable, was declining) -- missed case:")
print(false_negatives[review_cols].to_string(index=False))

False positives (model said declining, wasn't) -- confident but wrong, worth reading:
          content_id  rf_score  is_declining_label  days_since_last_update  avg_position  ctr  engagement_rate
content_0b47dae0c7f9  0.823734                   0                     103          23.1  0.0              0.0
content_884c401ce126  0.818140                   0                      92          23.1  0.0              0.0

False negative (model said stable, was declining) -- missed case:
          content_id  rf_score  is_declining_label  days_since_last_update  avg_position  ctr  engagement_rate
content_7bc32bc1df59  0.156644                   1                      92           0.0  0.0              0.0


**Why these are hard:** the two false positives share an almost identical profile — `avg_position ≈ 23.1`, `ctr = 0.0`, `engagement_rate = 0.0` — the model read "invisible on clicks and engagement" as a strong decline signal, and it usually is, but these two pages happened to be labeled stable this 30-day window anyway. That's a real limit: the model can only see correlated risk, not the actual cause, so a page that looks exactly like a decliner can simply not decline yet. The false negative is the mirror case: `avg_position = 0` (genuinely unranked, `has_avg_position = 0`) with zero recorded CTR/engagement — a page so far outside the "typical" pattern that the model reads it as noise rather than risk, yet it declined anyway. Both point at the same boundary: this feature set has nothing for whatever happened off-page (a competitor's refresh, an algorithm shift, a demand swing) — a limit of the lane's available signals, not a bug in either model.

## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere (only pseudonymous `content_id`/`client_id`)
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.